Cart-Pole program using Cross-Entropy Method.

In [1]:
import gym 
from collections import namedtuple
import numpy as np 
import torch 
import torch.nn as nn
import torch.optim as optim
import os

# Define constants for hidden layer size, batch size, and percentile for filtering
HIDDEN_SIZE = 128 
BATCH_SIZE = 16
PERCENTILE = 70

# Define a simple neural network class
class Net(nn.Module):
    def __init__(self, obs_size, hidden_size, n_actions):
        """
        Initializes the neural network with input size, hidden size, and number of actions.
        The network takes the observations as input and outputs action scores.
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden_size),  # First layer: Input size to hidden layer
            nn.ReLU(),                         # Activation function: ReLU
            nn.Linear(hidden_size, n_actions)  # Output layer: Hidden layer to action scores
        )

    def forward(self, x):
        # Forward pass through the network
        return self.net(x)

# Named tuples to store episodes and steps within each episode
Episode = namedtuple("Episode", field_names=["reward", "steps"])
EpisodeStep = namedtuple("EpisodeStep", field_names=["observation", "action"])

# Function to iterate through batches of episodes
def iterate_batches(env, net, batch_size):
    """
    Generates batches of episodes by interacting with the environment using the policy defined by the network.
    """
    batch = []  # List to store completed episodes
    episode_reward = 0.0  # Accumulates reward for each episode
    episode_steps = []  # List to store steps within the current episode
    obs = env.reset()  # Reset the environment to start a new episode
    sm = nn.Softmax(dim=1)  # Softmax function to convert network output to probabilities

    while True:
        # Convert observation to a tensor and get action probabilities
        obs_v = torch.FloatTensor([obs])
        act_probs_v = sm(net(obs_v))
        act_probs = act_probs_v.data.numpy()[0]  # Convert probabilities to a numpy array
        
        # Sample an action based on the action probabilities
        action = np.random.choice(len(act_probs), p=act_probs)
        
        # Take the action in the environment
        next_obs, reward, is_done, _ = env.step(action)

        # Accumulate reward and store the step
        episode_reward += reward
        step = EpisodeStep(observation=obs, action=action)
        episode_steps.append(step)

        # Check if the episode is done
        if is_done:
            # Store the episode and reset reward and steps
            e = Episode(reward=episode_reward, steps=episode_steps)
            batch.append(e)
            episode_reward = 0.0
            episode_steps = []
            next_obs = env.reset()  # Start a new episode
            # Yield the batch when it reaches the specified size
            if len(batch) == batch_size: 
                yield batch
                batch = []  # Reset batch for the next set of episodes

        obs = next_obs  # Update the observation for the next step

# Function to filter episodes based on reward percentile
def filter_batch(batch, percentile):
    """
    Filters episodes to keep only the top-performing ones based on rewards.
    Returns observations and actions from the best episodes.
    """
    rewards = list(map(lambda s: s.reward, batch))  # Extract rewards from the batch
    reward_bound = np.percentile(rewards, percentile)  # Determine the reward cutoff based on percentile
    reward_mean = float(np.mean(rewards))  # Calculate the mean reward

    train_obs = []  # List to store observations
    train_act = []  # List to store actions

    # Iterate through episodes and keep those with rewards above the cutoff
    for reward, steps in batch:
        if reward < reward_bound:
            continue 
        # Add the observations and actions from the selected episodes
        train_obs.extend(map(lambda step: step.observation, steps))
        train_act.extend(map(lambda step: step.action, steps))

    # Convert observations and actions to tensors
    train_obs_v = torch.FloatTensor(train_obs)
    train_act_v = torch.LongTensor(train_act)  # Convert actions to LongTensor (required by CrossEntropyLoss)
    return train_obs_v, train_act_v, reward_bound, reward_mean 

# Main script to train the model using the CartPole environment
if __name__ == "__main__":
    # Initialize the environment and wrap it with RecordVideo for recording
    env = gym.make("CartPole-v1")
    video_dir = "./video"  # Directory to save videos
    if not os.path.exists(video_dir):
        os.makedirs(video_dir)
    
    # Use gym.wrappers.RecordVideo to record videos of the environment
    env = gym.wrappers.RecordVideo(env, video_dir, episode_trigger=lambda episode_id: True)
    
    obs_size = env.observation_space.shape[0]  # Get the size of the observation space
    n_actions = env.action_space.n  # Get the number of actions

    # Initialize the neural network, loss function, and optimizer
    net = Net(obs_size, HIDDEN_SIZE, n_actions)
    objective = nn.CrossEntropyLoss()  # Cross-entropy loss for classification
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-2)  # Adam optimizer for training

    # Main training loop
    for iter_no, batch in enumerate(iterate_batches(env, net, BATCH_SIZE)):
        # Filter the batch to get top-performing episodes
        obs_v, acts_v, reward_b, reward_m = filter_batch(batch, PERCENTILE)
        
        # Reset gradients, compute loss, and perform backpropagation
        optimizer.zero_grad()
        action_scores_v = net(obs_v)  # Get action scores from the network
        loss_v = objective(action_scores_v, acts_v)  # Compute the loss
        loss_v.backward()  # Backpropagation to compute gradients
        optimizer.step()  # Update the network parameters
        
        # Print iteration number, loss, mean reward, and reward bound
        print("%d: loss=%.3f, reward_mean=%.1f, rw_bound=%.1f" % (
            iter_no, loss_v.item(), reward_m, reward_b))

        # Check if the mean reward exceeds the threshold to solve the problem
        if reward_m > 199:
            print("Solved!")
            break

    # Close the environment to finalize the video recording
    env.close()

    # To get confirmation of fininshed training and saved video
    print(f"Video saved in {video_dir}.")


C:\Users\utkri\anaconda3\envs\pytorch_env\Lib\site-packages\gym\wrappers\record_video.py:41: UserWarning: WARN: Overwriting existing videos at C:\Users\utkri\video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
C:\Users\utkri\AppData\Local\Temp\ipykernel_21560\3091460889.py:49: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:264.)
  obs_v = torch.FloatTensor([obs])


0: loss=0.693, reward_mean=16.8, rw_bound=19.5
1: loss=0.684, reward_mean=23.8, rw_bound=26.5
2: loss=0.676, reward_mean=32.7, rw_bound=42.0
3: loss=0.662, reward_mean=26.3, rw_bound=30.0
4: loss=0.650, reward_mean=36.3, rw_bound=40.5
5: loss=0.639, reward_mean=37.8, rw_bound=45.5
6: loss=0.616, reward_mean=46.9, rw_bound=55.0
7: loss=0.623, reward_mean=46.5, rw_bound=55.0
8: loss=0.600, reward_mean=65.5, rw_bound=77.0
9: loss=0.601, reward_mean=67.4, rw_bound=80.0
10: loss=0.613, reward_mean=52.0, rw_bound=59.5
11: loss=0.596, reward_mean=71.2, rw_bound=80.0
12: loss=0.599, reward_mean=61.8, rw_bound=69.5
13: loss=0.574, reward_mean=65.3, rw_bound=69.0
14: loss=0.591, reward_mean=75.9, rw_bound=86.5
15: loss=0.568, reward_mean=82.9, rw_bound=88.5
16: loss=0.593, reward_mean=87.3, rw_bound=97.0
17: loss=0.567, reward_mean=85.0, rw_bound=109.0
18: loss=0.572, reward_mean=87.8, rw_bound=104.0
19: loss=0.554, reward_mean=97.4, rw_bound=124.5
20: loss=0.566, reward_mean=140.2, rw_bound=179